# xHuBERT Experiments 2+7: HuBERT Frozen Baseline + Layer Probing
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

## Pipeline
```
Step 1: Exp2 -- HuBERT frozen + SVM/RF (5-fold + LOSGO)
Step 2: Exp7 -- Per-layer probing with logistic regression (LOSGO)
```

**Runtime**: **T4 GPU**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

WORK = "/content/drive/MyDrive/xhubert_results"
os.makedirs(WORK, exist_ok=True)

os.environ["XHUBERT_SAVE_DIR"] = WORK
os.environ["RAVDESS_ROOT"] = os.path.join(WORK, "RAVDESS")

print(f"Working directory: {WORK}")

In [ ]:
!pip install -q --upgrade transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
import os, subprocess

REPO_DIR = "/content/ravdess_experiment"

if os.path.exists(REPO_DIR):
    print("Repo exists, pulling latest ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("Cloning repo ...")
    subprocess.run([
        "git", "clone", "-b", "feature/xhubert-rewrite",
        "https://github.com/nhunet/ravdess_experiment.git", REPO_DIR
    ], check=True)

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

# Verify required files
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
    "experiments/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("All required files OK")

In [ ]:
import os

DRIVE_RAVDESS = os.environ["RAVDESS_ROOT"]

if not os.path.exists(DRIVE_RAVDESS):
    print("Not in Drive -> Download from Zenodo...")
    !wget -q --show-progress https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d /content/RAVDESS_tmp
    !mkdir -p "$DRIVE_RAVDESS"
    !cp -r /content/RAVDESS_tmp/* "$DRIVE_RAVDESS/"
    !rm -rf /content/RAVDESS_tmp Audio_Speech_Actors_01-24.zip
    print("Saved to Drive.")
else:
    import glob
    n = len(glob.glob(os.path.join(DRIVE_RAVDESS, "**/*.wav"), recursive=True))
    print(f"Already exist ({n} wav files), don't download.")

In [ ]:
import config
from utils import ensure_dirs

ensure_dirs()
print(f"SAVE_DIR:  {config.SAVE_DIR}")
print(f"CSV_DIR:   {config.CSV_DIR}")
print(f"FIG_DIR:   {config.FIG_DIR}")
print(f"CKPT_DIR:  {config.CKPT_DIR}")
print(f"EMB_DIR:   {config.EMB_DIR}")
print(f"LOG_DIR:   {config.LOG_DIR}")
print(f"RAVDESS:   {config.RAVDESS_ROOT}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset (16kHz for HuBERT)

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HUBERT)
dataset.print_summary()

## Exp2: HuBERT Frozen Baseline

In [ ]:
from experiments.exp2_hubert_frozen import run_exp2

df_exp2 = run_exp2(dataset=dataset, force=False)
print(df_exp2.groupby(["Model", "Protocol"])["accuracy"].agg(["mean", "std"]).round(2))

## Exp7: Layer Probing

In [ ]:
from experiments.exp7_layer_probing import run_exp7

df_exp7 = run_exp7(dataset=dataset, force=False)
print(df_exp7.groupby("Layer")["accuracy"].agg(["mean", "std"]).round(2))

## Visualization

In [ ]:
from visualization.plots import plot_exp7_layer_curve
import numpy as np, os, config

alpha_path = os.path.join(config.EMB_DIR, "layer_weights_fold0_seed42.npy")
alpha = np.load(alpha_path) if os.path.exists(alpha_path) else None

plot_exp7_layer_curve(df_exp7, alpha_weights=alpha)
print("Layer probing figure saved!")